In [ ]:
## %% [markdown]
# # 04. Building the final dataset
# Final stage: merging literature and experimental records, filling empty cells,
# consistency check, element statistics.



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
TABLES_DIR = BASE_DIR / "outputs" / "tables"

LIT_PATH = RAW_DIR / "database_row.xlsx"
EXP_PATH = RAW_DIR / "experimental_results.xlsx"
STATS_PATH = TABLES_DIR / "final_dataset_element_statistics.xlsx"
FINAL_XLSX_PATH = BASE_DIR / "database_final.xlsx"
FINAL_CSV_PATH  = BASE_DIR / "database_final.csv"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

ELEMENTS = ['Ni', 'Al', 'Co', 'Cr', 'Cu', 'Fe', 'Ti',
            'V', 'Mo', 'Hf', 'W', 'Ta', 'B', 'C', 'Nb', 'Mn']
ZERO_COLS = ["HT 1, °C", "HT 1, min", "Recrystallization, °C", "Recrystallization, min",
             "Aging 1, °C", "Aging 1, min", "Aging 2, °C", "Aging 2, min",
             "Cold deformation, %"]
NAN_COLS_NUM = ["Solvus Temperature, °C", "L12 fraction, %"]
NAN_COLS_TXT = ["Cooling_after_HT1", "Cooling_after_Recryst", "Cooling_after_Aging1"]
NAN_COLS = NAN_COLS_NUM + NAN_COLS_TXT

# Raise an error on L12 inconsistencies (True) or only report them (False)
STRICT_L12_CHECK = False


In [ ]:
# 1. Loading both files
df_lit = pd.read_excel(LIT_PATH)
df_exp = pd.read_excel(EXP_PATH)

for df_ in (df_lit, df_exp):
    df_.columns = [c.strip() for c in df_.columns]

print(f"Literature records: {len(df_lit)}, columns: {df_lit.shape[1]}")
print(f"Experimental records: {len(df_exp)}, columns: {df_exp.shape[1]}")


In [ ]:
# 2. Column consistency check (column order may differ, names must match)
only_lit = [c for c in df_lit.columns if c not in df_exp.columns]
only_exp = [c for c in df_exp.columns if c not in df_lit.columns]
assert not only_lit and not only_exp, (
    f"Column names differ.\nOnly in database_row: {only_lit}\n"
    f"Only in experimental_results: {only_exp}"
)

# Experimental columns follow the literature column order
df_exp = df_exp[df_lit.columns]


In [ ]:
# 3. Appending the experimental records below the literature table
df = pd.concat([df_lit, df_exp], ignore_index=True)
assert len(df) == len(df_lit) + len(df_exp)
assert list(df.columns) == list(df_lit.columns)

print(f"Merged dataset: {len(df)} records "
      f"({len(df_lit)} literature + {len(df_exp)} experimental), {df.shape[1]} columns")

needed = (ELEMENTS + ZERO_COLS + NAN_COLS +
          ['Paper', 'DOI', 'Phase composition', 'Target state', 'L12 presence'])
missing = [c for c in needed if c not in df.columns]
assert not missing, f"Missing columns: {missing}\nAvailable: {list(df.columns)}"


In [ ]:
# 4. Filling empty cells
# Cells containing only whitespace are treated as empty
fill_cols = ZERO_COLS + NAN_COLS
df[fill_cols] = df[fill_cols].replace(r"^\s*$", np.nan, regex=True)

# Numeric validation before filling
el_num = df[ELEMENTS].apply(pd.to_numeric, errors='coerce')
assert not (df[ELEMENTS].notna() & el_num.isna()).any().any(), "Non-numeric values in elements"
assert not el_num.isna().any().any(), \
    f"Empty element cells: {el_num.isna().sum()[lambda s: s > 0].to_dict()}"

pr_num = df[ZERO_COLS].apply(pd.to_numeric, errors='coerce')
assert not (df[ZERO_COLS].notna() & pr_num.isna()).any().any(), "Non-numeric values in heat-treatment parameters"

opt_num = df[NAN_COLS_NUM].apply(pd.to_numeric, errors='coerce')
assert not (df[NAN_COLS_NUM].notna() & opt_num.isna()).any().any(), "Non-numeric values in optional numeric fields"

# Processing steps that were not applied -> 0
df[ELEMENTS] = el_num
df[ZERO_COLS] = pr_num.fillna(0)
assert not df[ELEMENTS + ZERO_COLS].isna().any().any()

# L12 consistency check uses the numeric values BEFORE the visible "NaN" text is written
l12_frac = opt_num["L12 fraction, %"]


In [ ]:
# 5. Consistency check: L12 fraction vs L12 presence
l12_presence = pd.to_numeric(df["L12 presence"], errors="coerce")
assert not (df["L12 presence"].notna() & l12_presence.isna()).any(), \
    "Non-numeric values in 'L12 presence'"

# Rows with an unreported fraction (NaN) are excluded: comparisons with NaN are False
bad_zero = df[(l12_frac == 0) & (l12_presence == 1)]  # fraction measured as 0, but L12 is marked as present
bad_pos = df[(l12_frac > 0) & (l12_presence == 0)]    # fraction > 0, but L12 is marked as absent

print(f"L12 fraction = 0 with L12 presence = 1: {len(bad_zero)}")
print(f"L12 fraction > 0 with L12 presence = 0: {len(bad_pos)}")
if len(bad_zero):
    print("\nPapers with fraction = 0 and L12 present:")
    print(bad_zero[["Paper", "DOI"]].drop_duplicates().to_string(index=False))
if len(bad_pos):
    print("\nPapers with fraction > 0 and L12 absent:")
    print(bad_pos[["Paper", "DOI"]].drop_duplicates().to_string(index=False))

if STRICT_L12_CHECK:
    assert len(bad_zero) == 0 and len(bad_pos) == 0, "L12 fraction / presence inconsistencies found"


In [ ]:
# %%
# 6. Saving the final dataset
# 6a. XLSX for viewing: unreported values are shown as visible "NaN" text
df_xlsx = df.copy()
df_xlsx[NAN_COLS] = df_xlsx[NAN_COLS].astype(object).fillna("NaN")
df_xlsx.to_excel(FINAL_XLSX_PATH, index=False)
print(f"Saved: {FINAL_XLSX_PATH} | {df_xlsx.shape[0]} rows, {df_xlsx.shape[1]} columns")

# 6b. CSV for machine use: unreported values are truly empty, numeric columns stay numeric
df_csv = df.copy()
text_cols = df_csv.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    df_csv[c] = df_csv[c].map(lambda v: v.strip() if isinstance(v, str) else v)
df_csv.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8",
              sep=",", decimal=".", na_rep="", lineterminator="\n")
print(f"Saved: {FINAL_CSV_PATH} | {df_csv.shape[0]} rows, {df_csv.shape[1]} columns")

# 6c. Round-trip check: CSV and XLSX must contain the same data
chk_csv = pd.read_csv(FINAL_CSV_PATH)
chk_xlsx = pd.read_excel(FINAL_XLSX_PATH)       # "NaN" text is parsed as NaN by pandas
assert chk_csv.shape == chk_xlsx.shape == df.shape
assert list(chk_csv.columns) == list(df.columns)
num_cols = ELEMENTS + ZERO_COLS + NAN_COLS_NUM
pd.testing.assert_frame_equal(chk_csv[num_cols], chk_xlsx[num_cols],
                              check_dtype=False, atol=1e-9)
assert chk_csv[ELEMENTS + ZERO_COLS].notna().all().all()
print("CSV/XLSX consistency check passed")


In [ ]:
# 7. Element statistics of the final dataset
el_stats = pd.DataFrame({
    'Records with element (>0)': (df[ELEMENTS] > 0).sum(),
    'Zero (element absent)': (df[ELEMENTS] == 0).sum(),
    'Min (>0), at.%': df[ELEMENTS].where(df[ELEMENTS] > 0).min(),
    'Max, at.%': df[ELEMENTS].max()}).round(2)
el_stats['Share of records, %'] = (
    el_stats['Records with element (>0)'] / len(df) * 100
).round(1)

assert (el_stats['Records with element (>0)'] + el_stats['Zero (element absent)'] == len(df)).all()

el_stats = el_stats.loc[ELEMENTS].rename_axis('Element').reset_index()
print(el_stats.to_string(index=False))

with pd.ExcelWriter(STATS_PATH, engine='openpyxl') as writer:
    el_stats.to_excel(writer, sheet_name='Elements', index=False)
    ws = writer.sheets['Elements']
    for col_cells in ws.columns:
        width = max(len(str(c.value or '')) for c in col_cells)
        ws.column_dimensions[col_cells[0].column_letter].width = min(max(width + 3, 10), 60)
print(f"Statistics saved: {STATS_PATH}")

SOURCE = FINAL_XLSX_PATH
OUTPUT = TABLES_DIR / "final_dataset_statistics.xlsx"
ELEMENTS = ["Ni", "Al", "Co", "Cr", "Cu", "Fe", "Ti", "V", "Mo", "Hf", "W", "Ta", "B", "C", "Nb", "Mn"]
PROCESSING = ["HT 1, °C", "HT 1, min", "Recrystallization, °C", "Recrystallization, min", "Aging 1, °C", "Aging 1, min", "Aging 2, °C", "Aging 2, min", "Cold deformation, %"]
COOLING = {"Cooling_after_HT1": "HT 1, °C", "Cooling_after_Recryst": "Recrystallization, °C", "Cooling_after_Aging1": "Aging 1, °C"}
OPTIONAL = ["Solvus Temperature, °C", "L12 fraction, %"]
FLAGS = ["L12 presence", "Target state"]
df = pd.read_excel(SOURCE).replace(r"^\s*$", float("nan"), regex=True)
n = len(df)

def count_pct(k):
    return f"{k} ({100 * k / n:.1f}%)"

def value_range(s):
    s = s.dropna()
    return f"{s.min():g}–{s.max():g}" if len(s) else "—"

rows, audit, frequencies, cooling_rows = [], [], [], []
for col in df.columns:
    s = df[col]
    filled = int(s.notna().sum())
    zero = int(s.eq(0).sum())
    if col in ELEMENTS:
        group, selected = "Composition", s[s > 0]
        count, summary = len(selected), value_range(selected) + " at.% (non-zero)"
        interpretation = "Records with non-zero concentration"
    elif col in PROCESSING:
        group, selected = "Processing", s[s > 0]
        unit = "°C" if "°C" in col else "min" if "min" in col else "%"
        count, summary = len(selected), value_range(selected) + " " + unit + " (non-zero)"
        interpretation = "Records with non-zero processing parameter"
    elif col in COOLING:
        group, count = "Cooling", filled
        summary = f"{s.nunique()} reported categories"
        interpretation = "Records with reported cooling mode; denominator = all records"
        active = df[COOLING[col]].gt(0)
        k = int((active & s.notna()).sum())
        cooling_rows.append({"Field": col, "Step applied": int(active.sum()), "Mode reported, step applied": k, "Mode missing, step applied": int((active & s.isna()).sum()), "Step not applied": int((~active).sum()), "Mode reported, step not applied": int((~active & s.notna()).sum()), "Reported among applied, %": round(100*k/active.sum(), 1) if active.any() else None})
    elif col in OPTIONAL:
        group, count = "Target properties", filled
        unit = "°C" if "°C" in col else "%"
        summary = value_range(s) + " " + unit + " (including zero)"
        interpretation = "Records with non-missing property value; zero included"
    elif col in FLAGS:
        group, count = "Phase indicators", filled
        summary = f"1: {count_pct(int(s.eq(1).sum()))}; 0: {count_pct(zero)}"
        interpretation = "Records with defined flag; class counts in summary"
    elif col == "Phase composition":
        group, count = "Phase constitution", filled
        summary, interpretation = f"{s.nunique()} phase combinations", "Records with phase label"
    elif col == "Preparation":
        group, count = "Processing", filled
        summary, interpretation = f"{s.nunique()} preparation categories", "Records with preparation method"
    elif col == "Sum":
        group, count = "Composition check", filled
        summary, interpretation = value_range(s.round(6)) + " at.%", "Records with stored concentration sum"
    elif col == "Composition, at.%":
        group, count = "Identification", filled
        unique_numeric = df[ELEMENTS].round(3).drop_duplicates().shape[0]
        summary = f"{s.nunique()} formula strings; {unique_numeric} numerical compositions (3 decimals)"
        interpretation = "Records with composition formula"
    elif col == "Paper":
        group, count = "Source", filled
        lit = df["DOI"].astype(str).str.strip().ne("this work")
        summary = f"{df.loc[lit, col].nunique()} literature identifiers; {int((~lit).sum())} this-work records"
        interpretation = "Records with source/sample identifier"
    elif col == "DOI":
        group, count = "Source", filled
        literature = s.dropna().astype(str).str.strip()
        summary = f"{literature[literature.ne('this work')].nunique()} distinct literature DOI strings; 'this work': {int(literature.eq('this work').sum())} records"
        interpretation = "Non-empty entries; 'this work' is not a DOI"
    else:
        raise ValueError(f"Define semantics for {col}")
    rows.append({"Group": group, "Field": col, "Records, n (%)": count_pct(count), "Summary": summary})
    audit.append({"Field": col, "Non-missing": filled, "Missing": int(s.isna().sum()), "Zero": zero, "Distinct non-missing": int(s.nunique()), "Interpretation of table count": interpretation})
    if col in COOLING or col in FLAGS or col in ["Preparation", "Phase composition"]:
        for val, k in s.value_counts(dropna=False).items():
            frequencies.append({"Field": col, "Value": "NaN" if pd.isna(val) else str(val), "Records": int(k), "Share of all records, %": round(100*k/n, 1)})

notes = pd.DataFrame({"Notes": [
    f"All overall percentages use N = {n} records, not unique compositions.",
    "Element and processing ranges exclude zero. Counts refer to non-zero values, not non-missing entries.",
    "Zero in processing fields follows the dataset convention for a step not applied; this summary does not independently verify original sources.",
    "Property ranges and counts include valid zero values, notably L12 fraction.",
    "Cooling missingness is decomposed by step applicability on Cooling_context. Step applicability is inferred from temperature > 0.",
    "DOI='this work' is a source marker, not a valid DOI. Paper contains both publication identifiers and S1–S12 sample identifiers.",
    "Count of categorical values describes stored labels, not necessarily physically distinct routes.",
    "The all-fields table is an audit/overview draft and may need condensation for the manuscript."
]})
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT, engine="openpyxl") as writer:
    for name, table in {"Table4_all_fields": pd.DataFrame(rows), "Field_audit": pd.DataFrame(audit), "Category_counts": pd.DataFrame(frequencies), "Cooling_context": pd.DataFrame(cooling_rows), "Notes": notes}.items():
        table.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        ws.freeze_panes = "C2" if name == "Table4_all_fields" else "A2"
        ws.auto_filter.ref = ws.dimensions
        from openpyxl.styles import Alignment, Font, PatternFill
        for cell in ws[1]:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor="24465B")
        for cells in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0 for c in cells)
            ws.column_dimensions[cells[0].column_letter].width = min(max(width + 2, 14), 75)
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical="top", wrap_text=True)
            ws.row_dimensions[row[0].row].height = 32

In [ ]:
print(df["Paper"].map(type).value_counts())
print("NaN in df:", df["Paper"].isna().sum(), "of", len(df))
print("NaN in lit:", df_lit["Paper"].isna().sum(), "| exp:", df_exp["Paper"].isna().sum())
chk = pd.read_csv(FINAL_CSV_PATH)
print("NaN in csv:", chk["Paper"].isna().sum())